## Land Use from ESA WorldCover (Google Earth Engine)

This notebook fetches **ESA WorldCover v200** land cover and derives the **percentage of urban, agricultural, mining, and industrial** land within a **1 km radius** of each sample point (latitude, longitude).

### Prerequisites

1. **Install** the Earth Engine API: `pip install earthengine-api`
2. **Authenticate** (one-time): run the cell with `ee.Authenticate()`, then `ee.Initialize()`

### Dataset and class mapping

[ESA WorldCover 10 m v200](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200) has 11 land cover classes. We map to your requested categories:

| Category      | WorldCover class | Value | Description           |
|---------------|------------------|-------|-----------------------|
| **Urban**     | Built-up         | 50    | Built-up areas        |
| **Agricultural** | Cropland      | 40    | Cropland              |
| **Mining**    | Bare / sparse    | 60    | Bare soil, sparse veg (proxy for mining/quarries) |
| **Industrial**| Built-up         | 50    | Same as urban (WorldCover does not separate industrial) |

So **urban** and **industrial** both use class 50; we output both columns (identical) for compatibility. **Mining** uses class 60 as a proxy for exposed land.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os

import ee

### 1. Authenticate and initialize (run once)

Uncomment and run the authenticate cell once if needed, then run initialize.

In [ ]:
# One-time: opens browser to sign in with Google and grant Earth Engine access
# ee.Authenticate()

In [4]:
try:
    ee.Initialize()
    print("Earth Engine initialized.")
except Exception as e:
    print("Run ee.Authenticate() first, then ee.Initialize(). Error:", e)

Earth Engine initialized.


### 2. Load points (Latitude, Longitude)

Use a CSV with `Latitude` and `Longitude`. Paths are resolved relative to project root or `topography_features/`.

In [5]:
# Paths: relative to project root; if running from topography_features/, use parent dir
if os.path.exists("data"):
    DATA_DIR = "data"
else:
    DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
ORIGINAL_DIR = os.path.join(DATA_DIR, "original")

# Load locations
locations_path = os.path.join(PROCESSED_DIR, "elevation_gradient_locations.csv")
if not os.path.exists(locations_path):
    locations_path = os.path.join(ORIGINAL_DIR, "submission_template.csv")

df = pd.read_csv(locations_path)
locations = df[["Latitude", "Longitude"]].drop_duplicates().reset_index(drop=True)
print(f"Unique locations: {len(locations)}")

Unique locations: 162


### 3. WorldCover class codes (urban, agricultural, mining, industrial)

ESA WorldCover v200: 10=Tree, 20=Shrub, 30=Grass, 40=Cropland, 50=Built-up, 60=Bare/sparse, 70=Snow, 80=Water, 90=Wetland, 95=Mangroves, 100=Moss.

In [6]:
# Class values in WorldCover v200 (band 'Map')
CLASS_URBAN = 50          # Built-up
CLASS_AGRICULTURAL = 40   # Cropland

BUFFER_M = 1000           # 1 km radius
WORLDCOVER_SCALE = 10     # 10 m resolution

### 4. Build FeatureCollection of 1 km buffers

Each feature is a circle of radius 1 km around a sample point, with an `id` to match back to the dataframe.

In [7]:
def buffers_to_feature_collection(locations_df, radius_m=1000):
    """Build ee.FeatureCollection of circular buffers (lon, lat) with radius_m meters."""
    features = []
    for i, row in locations_df.iterrows():
        lon, lat = float(row["Longitude"]), float(row["Latitude"])
        pt = ee.Geometry.Point([lon, lat])
        buffer = pt.buffer(radius_m)
        feat = ee.Feature(buffer, {"id": i})
        features.append(feat)
    return ee.FeatureCollection(features)

buffers_fc = buffers_to_feature_collection(locations, radius_m=BUFFER_M)
print(f"FeatureCollection with {locations.shape[0]} buffers (radius={BUFFER_M} m).")

FeatureCollection with 162 buffers (radius=1000 m).


### 5. Load WorldCover and compute land cover histogram per buffer

We use `reduceRegions` with `frequencyHistogram` to get pixel counts per class within each 1 km circle. Scale is 10 m (WorldCover resolution).

In [ ]:
# ESA WorldCover 10 m v200 (single image from collection)
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")

# Per-buffer frequency histogram (pixel counts per class)
reduced = worldcover.reduRegions(
    collection=buffers_fc,
    reducer=ee.Reducer.frequencyHistogram(),
    scale=WORLDCOVER_SCALE
)

# Transfer to client (for large point sets, consider Export.table.toDrive)
result = reduced.getInfo()

TypeError: Image.reduceRegions() got an unexpected keyword argument 'maxPixels'

### 6. Parse histograms and compute percentages

Each feature has a property `Map` (band name) whose value is a dictionary: class value (as string) → pixel count. We sum all counts for total pixels, then compute percentage per category.

In [ ]:
def parse_histogram_and_percentages(result, band_name="Map"):
    """
    Parse getInfo() from reduceRegions(frequencyHistogram).
    Returns DataFrame with id, pct_urban, pct_agricultural, pct_mining, pct_industrial.
    """
    features = result.get("features", [])
    rows = []
    for f in features:
        props = f.get("properties", {})
        idx = props.get("id")
        hist = props.get(band_name)
        if hist is None:
            rows.append({
                "id": idx,
                "pct_urban": np.nan,
                "pct_agricultural": np.nan,
            })
            continue
        # Keys may be strings (e.g. '10', '40', '50', '60')
        total = sum(int(v) for v in hist.values())
        if total == 0:
            pct_urban = pct_ag = pct_mining = pct_ind = 0.0
        else:
            count_urban = int(hist.get(str(CLASS_URBAN), 0))
            count_ag = int(hist.get(str(CLASS_AGRICULTURAL), 0))
            pct_urban = 100.0 * count_urban / total
            pct_ag = 100.0 * count_ag / total
        rows.append({
            "id": idx,
            "pct_urban": pct_urban,
            "pct_agricultural": pct_ag,
        })
    out = pd.DataFrame(rows).sort_values("id").reset_index(drop=True)
    return out

In [ ]:
pct_df = parse_histogram_and_percentages(result)
locations["pct_urban"] = pct_df["pct_urban"].values
locations["pct_agricultural"] = pct_df["pct_agricultural"].values
locations.head(10)

### 7. Save and optionally merge with full dataset

Save unique locations with land use percentages. Optionally merge back into a full dataframe on `Latitude` and `Longitude`.

In [ ]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
out_path = os.path.join(PROCESSED_DIR, "land_use_worldcover_1km_locations.csv")
locations.to_csv(out_path, index=False)
print(f"Saved {len(locations)} rows to {out_path}")

In [ ]:
# Optional: merge land use percentages into the original dataframe
land_use_cols = ["Latitude", "Longitude", "pct_urban", "pct_agricultural", "pct_mining", "pct_industrial"]
df_merged = df.merge(
    locations[land_use_cols],
    on=["Latitude", "Longitude"],
    how="left",
)
df_merged.head()

### Notes

- **Urban vs industrial**: WorldCover has only "Built-up" (50). We output both `pct_urban` and `pct_industrial` using the same class; they are identical.
- **Mining**: WorldCover has no mining class. We use **Bare / sparse vegetation** (60) as a proxy for exposed land (quarries, mining, bare soil).
- **Buffer**: 1 km radius; area ≈ 3.14 km². Increase `BUFFER_M` if you want a larger radius.
- **Large point sets**: `getInfo()` can hit size/time limits. For many points, export with `ee.batch.Export.table.toDrive()` and load the table in a second step.